In [ ]:
!pip install transformers datasets torch accelerate peft bitsandbytes -q

In [29]:
pip install --upgrade accelerate transformers

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Looking in indexes: http://mirrors.aliyun.com/pypi/simple
Note: you may need to restart the kernel to use updated packages.


In [5]:
import os
os.environ["WANDB_DISABLED"] = "true"
os.environ['HF_ENDPOINT'] = 'https://hf-mirror.com'

import torch
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling,
)
from datasets import Dataset
from peft import LoraConfig, get_peft_model, TaskType
import random

# 检查GPU可用性
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"使用设备: {device}")
if torch.cuda.is_available():
    print(f"GPU型号: {torch.cuda.get_device_name(0)}")

# 古典诗词训练数据集
poetry_texts = [
    # 楚辞风格
    "离骚兮悲怨，屈子之忧思。香草美人兮，托物以言志。",
    "帝高阳之苗裔兮，朕皇考曰伯庸。摄提贞于孟陬兮，惟庚寅吾以降。",
    "路漫漫其修远兮，吾将上下而求索。饮余马于咸池兮，总余辔乎扶桑。",
    "惟草木之零落兮，恐美人之迟暮。不抚壮而弃秽兮，何不改乎此度。",
    "纷吾既有此内美兮，又重之以修能。扈江离与辟芷兮，纫秋兰以为佩。",

    # 唐诗风格
    "春眠不觉晓，处处闻啼鸟。夜来风雨声，花落知多少。",
    "白日依山尽，黄河入海流。欲穷千里目，更上一层楼。",
    "床前明月光，疑是地上霜。举头望明月，低头思故乡。",
    "红豆生南国，春来发几枝。愿君多采撷，此物最相思。",
    "独在异乡为异客，每逢佳节倍思亲。遥知兄弟登高处，遍插茱萸少一人。",

    # 宋词风格
    "明月几时有，把酒问青天。不知天上宫阙，今夕是何年。",
    "十年生死两茫茫，不思量，自难忘。千里孤坟，无处话凄凉。",
    "寻寻觅觅，冷冷清清，凄凄惨惨戚戚。乍暖还寒时候，最难将息。",
    "昨夜雨疏风骤，浓睡不消残酒。试问卷帘人，却道海棠依旧。",

    # 古风诗词
    "采菊东篱下，悠然见南山。山气日夕佳，飞鸟相与还。",
    "曲径通幽处，禅房花木深。山光悦鸟性，潭影空人心。",
    "青青子衿，悠悠我心。但为君故，沉吟至今。",
    "关关雎鸠，在河之洲。窈窕淑女，君子好逑。",
    "蒹葭苍苍，白露为霜。所谓伊人，在水一方。",
]

# 选择模型 - 使用较小的中文模型
# model files: https://hf-mirror.com/uer/gpt2-chinese-cluecorpussmall/tree/main
model_name = "./model_cache"

print("正在加载模型和分词器...")
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    # device_map="auto"
)

# 设置pad token
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# 配置LoRA参数
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    inference_mode=False,
    r=8,
    lora_alpha=16,
    lora_dropout=0.1,
    target_modules=["c_attn", "c_proj"]
)

# 应用LoRA
model = get_peft_model(model, lora_config)
print("可训练参数量:")
model.print_trainable_parameters()

# 简化的数据预处理 - 直接处理文本
def prepare_training_data(texts, max_length=64):
    """准备训练数据"""
    input_ids = []

    for text in texts:
        # 为每个诗词添加特殊格式
        formatted_text = f"创作古诗：{text}<|endoftext|>"

        # 分词
        tokens = tokenizer(
            formatted_text,
            truncation=True,
            max_length=max_length,
            padding=False,
            return_tensors=None
        )

        input_ids.append(tokens['input_ids'])

    return input_ids

# 准备训练数据
print("准备训练数据...")
train_input_ids = prepare_training_data(poetry_texts)

# 创建自定义数据集类
class PoetryDataset(torch.utils.data.Dataset):
    def __init__(self, input_ids):
        self.input_ids = input_ids

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        # 返回字典，不转换为tensor
        return {
            'input_ids': self.input_ids[idx]  # 保持为列表
        }

# 创建数据集
train_dataset = PoetryDataset(train_input_ids)
print(f"训练数据集大小: {len(train_dataset)}")

# 使用transformers的标准数据整理器
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False,  # 不使用masked language modeling
    pad_to_multiple_of=8,
)

# 训练参数
training_args = TrainingArguments(
    output_dir="./poetry-model",
    overwrite_output_dir=True,
    num_train_epochs=10,
    per_device_train_batch_size=4,  # 增加批次大小
    gradient_accumulation_steps=2,
    warmup_steps=10,
    logging_steps=5,
    save_steps=50,
    learning_rate=3e-4,
    fp16=True,
    dataloader_pin_memory=False,
    report_to="none",
)

# 创建训练器
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    data_collator=data_collator,
)

# 开始训练
print("开始训练模型...")
print("="*50)
trainer.train()

print("训练完成！")
print("="*50)

# 生成诗词的函数
def generate_poetry(prompt="创作古诗：", max_length=80, temperature=0.8, num_return_sequences=2):
    """生成古典诗词"""
    model.eval()

    # 编码输入
    inputs = tokenizer.encode(prompt, return_tensors="pt").to(device)

    # 生成文本
    with torch.no_grad():
        outputs = model.generate(
            inputs,
            max_length=max_length,
            temperature=temperature,
            num_return_sequences=num_return_sequences,
            do_sample=True,
            top_p=0.9,
            top_k=50,
            repetition_penalty=1.1,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id
        )

    # 解码结果
    results = []
    for output in outputs:
        text = tokenizer.decode(output, skip_special_tokens=True)
        # 移除提示部分
        if prompt in text:
            generated_text = text.replace(prompt, "").strip()
        else:
            generated_text = text.strip()
        results.append(generated_text)

    return results

# 测试生成功能
print("测试诗词生成功能：")
print("="*50)

test_prompts = [
    "创作古诗：春天",
    "创作古诗：月亮",
    "创作古诗：思乡",
    "创作古诗：山水",
    "创作古诗：离别"
]

for prompt in test_prompts:
    print(f"\n主题：{prompt}")
    print("-" * 30)

    try:
        poems = generate_poetry(prompt, max_length=60, num_return_sequences=2)
        for i, poem in enumerate(poems):
            if poem.strip():
                print(f"生成 {i+1}: {poem}")
    except Exception as e:
        print(f"生成错误: {e}")

print("\n" + "="*50)
print("微调完成！现在可以生成古典诗词了。")
print("="*50)

# 交互式生成示例
print("\n使用示例：")
print("poems = generate_poetry('创作古诗：秋天', max_length=80)")
print("for poem in poems:")
print("    print(poem)")

# 简单的交互函数
def simple_generate(theme):
    """简单生成函数"""
    prompt = f"创作古诗：{theme}"
    results = generate_poetry(prompt, max_length=60, num_return_sequences=1)
    return results[0] if results else "生成失败"

# 测试简单生成
print(f"\n测试简单生成 - 主题'梅花':")
print(simple_generate("梅花"))

使用设备: cuda
GPU型号: NVIDIA GeForce RTX 2080 Ti
正在加载模型和分词器...


/root/miniconda3/lib/python3.12/site-packages/peft/tuners/lora/layer.py:2174: UserWarning: fan_in_fan_out is set to False but the target module is `Conv1D`. Setting fan_in_fan_out to True.
  warnings.warn(
Detected kernel version 5.4.0, which is below the recommended minimum of 5.5.0; this can cause the process to hang. It is recommended to upgrade the kernel to the minimum version or higher.


可训练参数量:
trainable params: 811,008 || all params: 102,879,744 || trainable%: 0.7883
准备训练数据...
训练数据集大小: 19
开始训练模型...


Step,Training Loss
5,4.129100
10,3.912700
15,3.544800
20,3.305600
25,2.947600
30,2.673600


训练完成！
测试诗词生成功能：

主题：创作古诗：春天
------------------------------
生成 1: 创 作 古 诗 ： 春 天 杜 甫 诗 集 < > < > < > < > < < > < > < < > < > < > < < > < < > < < > < < > < < > < < < > < < > < <
生成 2: 创 作 古 诗 ： 春 天 山 色 未 歇 ， 应 似 初 春 时 。 【 解 释 】 此 时 春 风 来 ， 万 物 复 苏 。 【 注 释 】 《 诗 经 · 大 雅 · 中 庸 》 ： 春 暖 花 开 ， 正 是

主题：创作古诗：月亮
------------------------------
生成 1: 创 作 古 诗 ： 月 亮 | ？? | ？??????????????????????????????????????????
生成 2: 创 作 古 诗 ： 月 亮 | 我 的 心 愿 ， 我 的 情 绪 ， 我 的 思 念 。 ｜ | （ 原 创 ） 古 诗 选 （ 二 ） 月 亮 是 天 空 中 的 月 亮 ， 它 的 背 后 是 星 辰 。 我 知 道

主题：创作古诗：思乡
------------------------------
生成 1: 创 作 古 诗 ： 思 乡 ， 是 愿 我 们 的 故 事 还 能 写 出 这 样 的 文 章 来 。 思 乡 、 留 恋 ， 在 此 情 绪 中 。 是 《 长 恨 歌 》 中 的 一 句 话 。 诗 中 有 这 么 一
生成 2: 创 作 古 诗 ： 思 乡 ， 人 ， 无 有 。 （ 《 题 集 · 第 三 卷 · 一 句 》 ） 人 生 三 苦 ： 苦 难 、 哀 伤 、 苦 楚 、 喜 乐 、 失 意 、 不 甘 、 忧 愁 。 这 句 话 在

主题：创作古诗：山水
------------------------------
生成 1: 创 作 古 诗 ： 山 水 | 景 ， 何 处 去 ？ 山 是 人 的 魂 魄 。 山 之 所 以 大 ， 在 于 其 自 然 风 貌 与 气 候 变 化 的 原 因 。 当 今 人 们 喜 欢 把 山 居 建 筑 作 为 文 人
生成 2: 创 作 古 诗 ： 山 水 长 风 ， 松 柏 黄 昏

In [ ]:
！